In [ ]:
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.mask import mask
from shapely.geometry import box, mapping
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# --------------------------------------------------------------------
# Paths
# --------------------------------------------------------------------
BASE = Path("C:/Users/EmmaGolub/Desktop/MRoS_local/local_data/")
DEM_10M = BASE / "DEM_AOI_TNM_10m.tif"
DEM_1KM = BASE / "DEM_1km.tif"
DEM_1KM_CLIP = BASE / "DEM_1km_clipped.tif" # this is the final clipped and reprojected version of the DEM we will use

# --------------------------------------------------------------------
# 1. Load original DEM
# --------------------------------------------------------------------
with rasterio.open(DEM_10M) as src:
    print("Original CRS:", src.crs)
    print("Original resolution:", src.res)

# --------------------------------------------------------------------
# 2. Reproject DEM → UTM Zone 11N (EPSG:26911)
# --------------------------------------------------------------------
dst_crs = "EPSG:26911"

with rasterio.open(DEM_10M) as src:
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds
    )

    profile = src.profile.copy()
    profile.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height
    })

    dem_utm = np.empty((height, width), dtype=np.float32)

    reproject(
        source=src.read(1),
        destination=dem_utm,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear
    )

print("Reprojection complete.")

# --------------------------------------------------------------------
# 3. Resample to 1 km resolution (averaging)
# --------------------------------------------------------------------
orig_res_x = abs(profile["transform"].a)
orig_res_y = abs(profile["transform"].e)
target_res = 1000  # meters

scale_x = orig_res_x / target_res
scale_y = orig_res_y / target_res

new_width  = int(width * scale_x)
new_height = int(height * scale_y)

dst_transform = rasterio.Affine(
    target_res, 0, profile["transform"].c,
    0, -target_res, profile["transform"].f
)

dem_1km = np.empty((new_height, new_width), dtype=np.float32)

reproject(
    source=dem_utm,
    destination=dem_1km,
    src_transform=profile["transform"],
    src_crs=dst_crs,
    dst_transform=dst_transform,
    dst_crs=dst_crs,
    resampling=Resampling.average
)

km_profile = {
    "driver": "GTiff",
    "height": new_height,
    "width": new_width,
    "count": 1,
    "dtype": "float32",
    "crs": dst_crs,
    "transform": dst_transform,
    "compress": "lzw"
}

if DEM_1KM.exists():
    DEM_1KM.unlink()

with rasterio.open(DEM_1KM, "w", **km_profile) as dst:
    dst.write(dem_1km, 1)

print("Saved:", DEM_1KM)

# --------------------------------------------------------------------
# 4. Compute SAFE INTERIOR CLIP BOUNDS
# --------------------------------------------------------------------
xmin = 162846.256115
ymin = 4175439.735232
xmax = 303272.041375
ymax = 4425513.051449

clip_poly = box(xmin, ymin, xmax, ymax)

print("Using custom clip bounds:")
print("xmin:", xmin)
print("ymin:", ymin)
print("xmax:", xmax)
print("ymax:", ymax)

# --------------------------------------------------------------------
# 5. Clip DEM to custom bounding box
# --------------------------------------------------------------------
with rasterio.open(DEM_1KM) as src:
    out_image, out_transform = mask(
        src,
        [mapping(clip_poly)],
        crop=True,
        filled=True,
        nodata=None
    )

    out_profile = src.profile.copy()
    out_profile.update({
        "height": out_image.shape[1],
        "width":  out_image.shape[2],
        "transform": out_transform,
        "dtype": "float32",
        "nodata": None
    })

if DEM_1KM_CLIP.exists():
    DEM_1KM_CLIP.unlink()

with rasterio.open(DEM_1KM_CLIP, "w", **out_profile) as dst:
    dst.write(out_image[0], 1)

print(f"Saved DEM clipped to custom bounds → {DEM_1KM_CLIP}")


# --------------------------------------------------------------------
# 6. Diagnostics
# --------------------------------------------------------------------
arr = out_image[0].astype(float)

print("\n--- Final DEM Diagnostics (should have ZERO NaNs) ---")
print("Shape:", arr.shape)
print("NaNs:", np.isnan(arr).sum())
print("Min/Max:", np.nanmin(arr), np.nanmax(arr))


In [ ]:
import matplotlib.pyplot as plt
from rasterio.plot import show
import rasterio

with rasterio.open(DEM_1KM_CLIP) as src:
    fig, ax = plt.subplots(figsize=(10, 8))
    show(src, ax=ax, cmap="terrain")   # <-- THIS ACTUALLY PLOTS THE RASTER
    ax.set_title("Clipped, Reprojected DEM")

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

dem_path = DEM_1KM_CLIP  # path object from above

with rasterio.open(dem_path) as src:
    arr = src.read(1).astype(float)
    transform = src.transform
    crs = src.crs
    bounds = src.bounds
    res = (abs(transform.a), abs(transform.e))

print("=====================================================")
print(" DEM Diagnostics")
print("=====================================================")
print(f"File: {dem_path}")
print(f"CRS: {crs}")
print(f"Resolution: {res[0]:.2f} x {res[1]:.2f} m")
print(f"Shape: {arr.shape}  (rows x cols)")
print(f"Bounds: {bounds}")

# Basic stats
print("\n--- Basic Statistics ---")
print(f"Min: {np.nanmin(arr):.2f}")
print(f"Max: {np.nanmax(arr):.2f}")
print(f"Mean: {np.nanmean(arr):.2f}")
print(f"Std Dev: {np.nanstd(arr):.2f}")

# NaN / Inf checks
print("\n--- Data Integrity ---")
print(f"NaNs:   {np.isnan(arr).sum():,}")
print(f"+Inf:   {np.isposinf(arr).sum():,}")
print(f"-Inf:   {np.isneginf(arr).sum():,}")

# Unique values (optional, careful with big rasters)
uniq_vals = np.unique(arr[~np.isnan(arr)])
print(f"\nUnique values (clipped sample): {uniq_vals[:10]} ... (total {len(uniq_vals)})")
